In [15]:
#setup
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Proyek_CRM_KELOMPOK')
sys.path.append(str(PROJECT_DIR))

from project_config import FINAL_DATA_PATH, OUTPUT_DIR, REPORT_DIR

!pip install streamlit pyngrok -q

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "36gxZKwQr3FBXF99XG7TieAQJoE_5DBoEHWmG2NRnzgpd9CLx"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print('Authtoken berhasil diset')

Authtoken berhasil diset


In [17]:
#app.py
app_path = PROJECT_DIR / 'app.py'

app_code = r"""
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

st.set_page_config(
    page_title='CRM Food Delivery Dashboard',
    layout='wide'
)

FINAL_DATA_PATH = "__FINAL_DATA_PATH__"
OUTPUT_DIR = "__OUTPUT_DIR__"

df = pd.read_csv(FINAL_DATA_PATH)
model_result = pd.read_csv(OUTPUT_DIR + '/model_comparison_result.csv')
feature_importance = pd.read_csv(OUTPUT_DIR + '/feature_importance_result.csv')
crm_strategy = pd.read_csv(OUTPUT_DIR + '/crm_strategy_matrix.csv')
customer_priority = pd.read_csv(OUTPUT_DIR + '/customer_priority_table.csv')

st.title('CRM Dashboard Food Delivery')
st.write('Prediksi risiko churn, segmentasi risiko, feature importance, dan strategi CRM.')

def pct(value):
    return f'{value:.1f}%'

def get_top_value(data, column):
    if column not in data.columns or len(data) == 0:
        return '-'
    value_counts = data[column].dropna().value_counts()
    if len(value_counts) == 0:
        return '-'
    return value_counts.index[0]

def get_top_count(data, column):
    if column not in data.columns or len(data) == 0:
        return 0
    value_counts = data[column].dropna().value_counts()
    if len(value_counts) == 0:
        return 0
    return int(value_counts.iloc[0])

def make_output_insight(data):
    if len(data) == 0:
        return 'Tidak ada data pada filter ini. Ubah filter untuk melihat insight.'

    total = len(data)
    risk_count = int((data['Output'] == 'No').sum())
    safe_count = int((data['Output'] == 'Yes').sum())
    risk_rate = risk_count / total * 100

    return (
        f'Dari {total} pelanggan terfilter, {risk_count} pelanggan atau {risk_rate:.1f}% masuk kategori risiko churn. '
        f'Artinya, fokus CRM perlu diarahkan ke pelanggan Output No karena kelompok ini berpotensi tidak membeli ulang. '
        f'Sementara {safe_count} pelanggan masih menunjukkan kecenderungan beli ulang.'
    )

def make_segment_insight(data):
    if len(data) == 0:
        return 'Tidak ada data pada filter ini. Ubah filter untuk melihat insight.'

    top_segment = get_top_value(data, 'risk_segment')
    top_count = get_top_count(data, 'risk_segment')
    total = len(data)
    top_percent = top_count / total * 100

    if top_segment == 'High Risk':
        action = 'Prioritas utama adalah campaign retensi cepat, seperti voucher fast delivery, kompensasi delay, dan pesan personal.'
    elif top_segment == 'Medium Risk':
        action = 'Kelompok ini masih berpeluang dipertahankan melalui loyalty point, promo personal, dan rekomendasi restoran.'
    elif top_segment == 'Low Risk':
        action = 'Kelompok ini cocok untuk program referral, membership, dan reward review positif.'
    else:
        action = 'Gunakan segmen risiko untuk menentukan prioritas campaign.'

    return (
        f'Segmen terbesar pada data terfilter adalah {top_segment} dengan {top_count} pelanggan atau {top_percent:.1f}%. '
        f'So what: {action}'
    )

def make_model_insight(model_df):
    if len(model_df) == 0:
        return 'Belum ada hasil evaluasi model.'

    best_model = model_df.sort_values('f1_score', ascending=False).iloc[0]

    return (
        f'Model dengan F1-score tertinggi adalah {best_model["model"]} dengan F1-score {best_model["f1_score"]:.3f}. '
        f'F1-score penting karena data memiliki kelas risiko yang lebih kecil. '
        f'Artinya, model tidak hanya dinilai dari akurasi, tetapi juga dari kemampuan menangkap pelanggan berisiko.'
    )

def make_feature_insight(feature_df):
    if len(feature_df) == 0:
        return 'Belum ada feature importance.'

    top_feature = feature_df.iloc[0]['feature_clean']
    top_importance = feature_df.iloc[0]['importance']

    return (
        f'Faktor paling penting menurut model adalah {top_feature} dengan nilai importance {top_importance:.3f}. '
        f'So what: faktor ini perlu menjadi dasar penyusunan strategi CRM karena paling membantu model membedakan pelanggan berisiko dan tidak berisiko.'
    )

def make_priority_insight(data):
    if len(data) == 0:
        return 'Tidak ada pelanggan pada filter ini.'

    top_customer = data.sort_values('risk_score', ascending=False).iloc[0]
    name = top_customer.get('customer_name', 'Pelanggan prioritas')
    score = top_customer['risk_score']
    segment = top_customer['risk_segment']
    reason = top_customer['reason_code']
    action = top_customer['crm_action']

    return (
        f'Pelanggan prioritas tertinggi adalah {name} dengan risk score {score:.3f}, segmen {segment}, dan alasan utama {reason}. '
        f'So what: tindakan CRM yang disarankan adalah {action}.'
    )

def make_map_insight(data):
    if len(data) == 0:
        return 'Tidak ada titik lokasi pada filter ini.'

    return (
        f'Peta menampilkan sebaran {len(data)} pelanggan sesuai filter. '
        f'So what: area dengan pelanggan berisiko tinggi dapat dipakai untuk prioritas campaign lokal, rekomendasi restoran terdekat, dan optimasi layanan delivery.'
    )

def make_strategy_insight(data):
    return (
        'CRM Strategy Matrix menghubungkan risk segment, karakteristik pelanggan, strategi, channel, dan KPI. '
        'So what: strategi tidak dibagikan sama rata ke semua pelanggan, tetapi disesuaikan dengan risiko dan alasan churn.'
    )

total_customer = len(df)
actual_risk_rate = df['churn_risk'].mean() * 100
high_risk = (df['risk_segment'] == 'High Risk').sum()
medium_risk = (df['risk_segment'] == 'Medium Risk').sum()
low_risk = (df['risk_segment'] == 'Low Risk').sum()

col1, col2, col3, col4, col5 = st.columns(5)
col1.metric('Total Customer', total_customer)
col2.metric('Actual Risk Rate', pct(actual_risk_rate))
col3.metric('High Risk', high_risk)
col4.metric('Medium Risk', medium_risk)
col5.metric('Low Risk', low_risk)

st.info(
    f'Dashboard ini memetakan {total_customer} pelanggan. '
    f'Actual risk rate sebesar {actual_risk_rate:.1f}% menunjukkan proporsi pelanggan yang berisiko tidak membeli ulang. '
    f'So what: pelanggan High Risk dan Medium Risk perlu menjadi prioritas retensi.'
)

st.divider()

st.sidebar.header('Filter Dashboard')

risk_options = ['All'] + sorted(df['risk_segment'].dropna().unique().tolist())
segment_filter = st.sidebar.selectbox('Risk Segment', risk_options)

reason_options = ['All'] + sorted(df['reason_code'].dropna().unique().tolist())
reason_filter = st.sidebar.selectbox('Reason Code', reason_options)

if 'Gender' in df.columns:
    gender_options = ['All'] + sorted(df['Gender'].dropna().unique().tolist())
    gender_filter = st.sidebar.selectbox('Gender', gender_options)
else:
    gender_filter = 'All'

if 'Occupation' in df.columns:
    occupation_options = ['All'] + sorted(df['Occupation'].dropna().unique().tolist())
    occupation_filter = st.sidebar.selectbox('Occupation', occupation_options)
else:
    occupation_filter = 'All'

min_score = st.sidebar.slider(
    'Minimal Risk Score',
    min_value=0.0,
    max_value=1.0,
    value=0.0,
    step=0.05
)

top_n = st.sidebar.slider(
    'Jumlah Pelanggan Ditampilkan',
    min_value=5,
    max_value=100,
    value=20,
    step=5
)

filtered_df = df.copy()

if segment_filter != 'All':
    filtered_df = filtered_df[filtered_df['risk_segment'] == segment_filter]

if reason_filter != 'All':
    filtered_df = filtered_df[filtered_df['reason_code'] == reason_filter]

if gender_filter != 'All':
    filtered_df = filtered_df[filtered_df['Gender'] == gender_filter]

if occupation_filter != 'All':
    filtered_df = filtered_df[filtered_df['Occupation'] == occupation_filter]

filtered_df = filtered_df[filtered_df['risk_score'] >= min_score]

st.subheader('Ringkasan Data Terfilter')

fcol1, fcol2, fcol3 = st.columns(3)
fcol1.metric('Jumlah Customer Terfilter', len(filtered_df))

if len(filtered_df) > 0:
    fcol2.metric('Rata-rata Risk Score', f'{filtered_df["risk_score"].mean():.3f}')
    fcol3.metric('Risk Score Tertinggi', f'{filtered_df["risk_score"].max():.3f}')
else:
    fcol2.metric('Rata-rata Risk Score', '0')
    fcol3.metric('Risk Score Tertinggi', '0')

if len(filtered_df) > 0:
    top_reason = get_top_value(filtered_df, 'reason_code')
    top_reason_count = get_top_count(filtered_df, 'reason_code')
    st.info(
        f'Filter saat ini menghasilkan {len(filtered_df)} pelanggan. '
        f'Reason code paling dominan adalah {top_reason} dengan {top_reason_count} pelanggan. '
        f'So what: campaign CRM sebaiknya mengikuti alasan risiko paling dominan pada data terfilter.'
    )
else:
    st.warning('Tidak ada pelanggan pada kombinasi filter ini.')

left, right = st.columns(2)

with left:
    st.subheader('Distribusi Output')
    output_count = filtered_df['Output'].value_counts()

    fig, ax = plt.subplots(figsize=(6, 4))
    output_count.plot(kind='bar', ax=ax)
    ax.set_xlabel('Output')
    ax.set_ylabel('Jumlah pelanggan')
    ax.set_title('Distribusi Output')
    st.pyplot(fig)

    st.info(make_output_insight(filtered_df))

with right:
    st.subheader('Distribusi Risk Segment')
    segment_count = filtered_df['risk_segment'].value_counts()

    fig, ax = plt.subplots(figsize=(6, 4))
    segment_count.plot(kind='bar', ax=ax)
    ax.set_xlabel('Risk segment')
    ax.set_ylabel('Jumlah pelanggan')
    ax.set_title('Distribusi Risk Segment')
    st.pyplot(fig)

    st.info(make_segment_insight(filtered_df))

st.subheader('Model Comparison')
st.dataframe(model_result, use_container_width=True, hide_index=True)
st.info(make_model_insight(model_result))

st.subheader('Top Feature Importance')
top_features = feature_importance.head(15)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_features['feature_clean'], top_features['importance'])
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('Top Feature Importance')
st.pyplot(fig)

st.info(make_feature_insight(feature_importance))

st.subheader('Customer Priority Table')

priority_cols = [
    'customer_id',
    'customer_name',
    'risk_score',
    'risk_segment',
    'reason_code',
    'crm_action',
    'crm_channel',
    'crm_kpi',
    'Age',
    'Gender',
    'Marital Status',
    'Occupation',
    'Monthly Income'
]

available_cols = [col for col in priority_cols if col in filtered_df.columns]

priority_view = (
    filtered_df[available_cols]
    .sort_values('risk_score', ascending=False)
    .head(top_n)
    .reset_index(drop=True)
)

st.dataframe(
    priority_view,
    use_container_width=True,
    hide_index=True
)

st.info(make_priority_insight(priority_view))

csv_download = priority_view.to_csv(index=False).encode('utf-8')

st.download_button(
    label='Download Customer Priority CSV',
    data=csv_download,
    file_name='filtered_customer_priority.csv',
    mime='text/csv'
)

if 'latitude' in filtered_df.columns and 'longitude' in filtered_df.columns:
    st.subheader('Peta Lokasi Customer')
    map_df = filtered_df[['latitude', 'longitude']].dropna()
    map_df = map_df.rename(columns={'latitude': 'lat', 'longitude': 'lon'})

    if len(map_df) > 0:
        st.map(map_df)
        st.info(make_map_insight(map_df))
    else:
        st.info('Tidak ada data lokasi untuk filter ini.')

st.subheader('CRM Strategy Matrix')
st.dataframe(crm_strategy, use_container_width=True, hide_index=True)
st.info(make_strategy_insight(crm_strategy))

st.caption('Nama pelanggan pada dashboard adalah data dummy untuk kebutuhan simulasi CRM. Nama dummy tidak dipakai sebagai fitur model.')
"""

app_code = app_code.replace("__FINAL_DATA_PATH__", str(FINAL_DATA_PATH))
app_code = app_code.replace("__OUTPUT_DIR__", str(OUTPUT_DIR))

with open(app_path, 'w') as f:
    f.write(app_code)

print('app.py berhasil dibuat:', app_path)

app.py berhasil dibuat: /content/drive/MyDrive/Proyek_CRM_KELOMPOK/app.py


In [18]:
# run streamlit
!streamlit run "{app_path}" --server.port 8501 &> /content/streamlit_logs.txt &

In [19]:
# buat link ngrok
from pyngrok import ngrok

public_url = ngrok.connect(8501, "http")
print('Link dashboard:', public_url)

Link dashboard: NgrokTunnel: "https://nonmutinous-nasir-inkless.ngrok-free.dev" -> "http://localhost:8501"


In [20]:
# simpan catatan dashboard
dashboard_notes = f"""
Dashboard Streamlit berhasil dibuat.

File app:
{app_path}

Cara menjalankan ulang:
1. Buka notebook 08_dashboard_streamlit.ipynb
2. Jalankan cell install Streamlit
3. Jalankan cell streamlit run
4. Jalankan cell ngrok
5. Buka link dashboard

Isi dashboard:
1. KPI total customer
2. Actual risk rate
3. High Risk, Medium Risk, Low Risk
4. Distribusi Output
5. Distribusi Risk Segment
6. Model comparison
7. Feature importance
8. Customer priority table
9. Map pelanggan
10. CRM strategy matrix
"""

notes_path = REPORT_DIR / 'dashboard_run_notes.txt'

with open(notes_path, 'w') as f:
    f.write(dashboard_notes)

print('Catatan dashboard disimpan:', notes_path)

Catatan dashboard disimpan: /content/drive/MyDrive/Proyek_CRM_KELOMPOK/reports/dashboard_run_notes.txt


In [ ]:
from pyngrok import ngrok

ngrok.kill()
print('Ngrok lama sudah dimatikan')

Ngrok lama sudah dimatikan
